# PyTorch Complete Reference Guide
### From Fundamentals to CNNs and Beyond — with Deep Learning Concepts Explained

This notebook is a self-contained, hands-on reference. Each section pairs a **concept explanation** with **runnable PyTorch code**. Run cells top to bottom the first time; after that, jump to whatever section you need.

**Roadmap**
1. Setup & Tensors
2. Autograd (how backpropagation actually works)
3. Building Neural Networks (`nn.Module`)
4. Loss Functions & Optimizers
5. The Training Loop (manual → clean)
6. Datasets & DataLoaders
7. Full Pipeline: Training an NN on Image Data
8. Convolutional Neural Networks (CNNs) — theory + code
9. Regularization: Dropout, BatchNorm, Weight Decay, Augmentation
10. Learning Rate Scheduling
11. Transfer Learning with Pretrained Models
12. GPU / Device Management
13. Saving & Loading Models
14. Advanced Training: Mixed Precision, Gradient Clipping, Accumulation
15. A Peek Beyond CNNs: RNNs & Transformers
16. Debugging Checklist & Cheat Sheet


## 1. Setup & Tensors

**Concept:** A `Tensor` is PyTorch's core data structure — an n-dimensional array like a NumPy array, but it can live on a GPU and track gradients automatically. Everything in deep learning (inputs, weights, activations, gradients) is a tensor.

In [1]:
# if using a non-conda environment
!python -m pip install <xyz> -U --force-reinstall
# If using a conda environment
!conda install --name <environment name> <xyz> --update-deps --force-reinstall

The system cannot find the file specified.


< was unexpected at this time.


In [2]:
!pip3 install torch torchvision torchaudio
%pip install numpy pandas matplotlib scikit-learn seaborn jupyterlab

Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.13.0+cpu
CUDA available: False


### Creating tensors

In [11]:
# From data
a = torch.tensor([1, 2, 3])
b = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
c = torch.tensor([[5,6], [7,8]])

# Common constructors
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
rand = torch.rand(2, 3)          # uniform [0, 1)
randn = torch.randn(2, 3)        # standard normal
arange = torch.arange(0, 10, 2)  # like range()
eye = torch.eye(3)               # identity matrix

# From / to NumPy (shares memory on CPU!)
np_arr = np.array([1, 2, 3])
t_from_np = torch.from_numpy(np_arr)
back_to_np = t_from_np.numpy()

print(a.shape, a.dtype, a.device)
print(b)
print(c)

torch.Size([3]) torch.int64 cpu
tensor([[1., 2.],
        [3., 4.]])
tensor([[5, 6],
        [7, 8]])


### Key tensor attributes
- `.shape` / `.size()` — dimensions
- `.dtype` — data type (`torch.float32` is the default for NN weights)
- `.device` — `cpu` or `cuda:0`
- `.requires_grad` — whether PyTorch should track operations for gradients (Section 2)

In [8]:
x = torch.randn(3, 4, dtype=torch.float32)
print("shape:", x.shape)
print("ndim:", x.ndim)
print("numel:", x.numel())  # total number of elements

shape: torch.Size([3, 4])
ndim: 2
numel: 12


### Operations, indexing, reshaping, broadcasting

**Broadcasting rule (same as NumPy):** dimensions are compared from the right; they're compatible if equal, or one of them is 1.

In [9]:
x = torch.randn(2, 3)
y = torch.randn(2, 3)

# Elementwise
print(x + y)
print(x * y)          # elementwise multiply (NOT matrix multiply)
print(torch.exp(x))

# Matrix multiplication
A = torch.randn(2, 3)
B = torch.randn(3, 4)
C = A @ B              # same as torch.matmul(A, B)
print("A @ B shape:", C.shape)

# Reshaping
r = torch.arange(12)
print(r.reshape(3, 4))
print(r.view(3, 4))    # view shares memory, reshape may copy if needed
print(r.reshape(3, 4).T)           # transpose (2D)
print(r.reshape(2, 2, 3).permute(2, 0, 1).shape)  # general transpose

# Indexing / slicing works like NumPy
m = torch.arange(9).reshape(3, 3)
print(m[0])       # first row
print(m[:, 1])    # second column
print(m[1:, :2])  # sub-matrix

# Broadcasting example
a = torch.ones(3, 1)
b = torch.ones(1, 4)
print((a + b).shape)  # -> (3, 4)

# Squeeze / unsqueeze (add or remove size-1 dims — very common for batching)
v = torch.randn(5)
print(v.unsqueeze(0).shape)  # (1, 5) -> add a batch dimension
print(v.unsqueeze(0).squeeze(0).shape)  # back to (5,)

tensor([[-0.3919,  0.8682, -1.5338],
        [-0.5843,  1.0693, -1.3205]])
tensor([[-0.2033,  0.0140,  0.1375],
        [-1.1901, -0.2367, -2.0201]])
tensor([[0.5028, 1.0166, 0.2373],
        [2.3099, 3.5166, 0.1078]])
A @ B shape: torch.Size([2, 4])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([[ 0,  4,  8],
        [ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11]])
torch.Size([3, 2, 2])
tensor([0, 1, 2])
tensor([1, 4, 7])
tensor([[3, 4],
        [6, 7]])
torch.Size([3, 4])
torch.Size([1, 5])
torch.Size([5])


> **Tip:** In-place ops end with an underscore, e.g. `x.add_(1)`. Avoid them on tensors that require gradients — they can break autograd's history.

## 2. Autograd — How Backpropagation Actually Works

**Concept:** When a tensor has `requires_grad=True`, PyTorch builds a **computational graph** as you perform operations on it. Calling `.backward()` on a scalar output walks that graph backward (reverse-mode automatic differentiation) and fills in `.grad` for every leaf tensor — this *is* backpropagation.

Think of it as: forward pass builds the graph → `loss.backward()` computes `d(loss)/d(param)` for every parameter in one call.

In [14]:
# A tiny manual example: y = w*x + b, loss = (y - target)^2
x = torch.tensor(2.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
target = torch.tensor(5.0)

y = w * x + b
loss = (y - target) ** 2

loss.backward()          # computes d(loss)/dw and d(loss)/db

print("y:", y.item())
print("loss:", loss.item())
print("dloss/dw:", w.grad.item())   # matches 2*(y-target)*x by hand
print("dloss/db:", b.grad.item())   # matches 2*(y-target)*1 by hand

y: 2.0
loss: 9.0
dloss/dw: -12.0
dloss/db: -6.0


### Important rules of autograd
- Gradients **accumulate** by default — you must zero them each step (`optimizer.zero_grad()` or `tensor.grad = None`).
- Only **float** tensors can require grad.
- `.backward()` only works directly on a **scalar** (like a loss). For non-scalars you'd pass a `gradient=` argument.
- Wrap inference code in `with torch.no_grad():` to skip graph-building and save memory/compute.
- `.detach()` returns a tensor that shares data but is cut off from the graph.

In [11]:
w.grad.zero_()
b.grad.zero_()

# Gradients accumulate if you forget to zero them:
for _ in range(3):
    y = w * x + b
    loss = (y - target) ** 2
    loss.backward()
    print("grad after this backward call:", w.grad.item())
    w.grad.zero_()  # reset — try commenting this out to see accumulation

# no_grad for inference
with torch.no_grad():
    y_pred = w * x + b   # no graph built, faster, no .grad tracking
print(y_pred.requires_grad)

grad after this backward call: -12.0
grad after this backward call: -12.0
grad after this backward call: -12.0
False


## 3. Building Neural Networks with `nn.Module`

**Concept:** A neural network is a stack of parameterized functions (layers) with non-linear **activation functions** between them. Without non-linearities, stacking linear layers collapses into a single linear layer — depth would be pointless.

Every custom network subclasses `nn.Module` and implements:
- `__init__`: define the layers (this is where parameters get created)
- `forward`: define how data flows through those layers

In [15]:
class SimpleNN(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, out_features)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)   # no activation here -> raw logits, common for classification
        return x

model = SimpleNN(in_features=10, hidden=32, out_features=3)
print(model)

dummy_input = torch.randn(4, 10)   # batch of 4 samples, 10 features each
output = model(dummy_input)
print("output shape:", output.shape)   # (4, 3) -> 4 samples, 3 class scores

SimpleNN(
  (fc1): Linear(in_features=10, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=3, bias=True)
  (relu): ReLU()
)
output shape: torch.Size([4, 3])


### Common layers & activations
| Layer | Purpose |
|---|---|
| `nn.Linear(in, out)` | Fully-connected layer: `y = xW^T + b` |
| `nn.Conv2d` | Spatial feature extraction (Section 8) |
| `nn.ReLU` | `max(0, x)` — cheap, avoids vanishing gradients vs sigmoid |
| `nn.Sigmoid` | Squashes to (0,1) — used for binary output / gates |
| `nn.Tanh` | Squashes to (-1,1) — zero-centered |
| `nn.Softmax(dim=-1)` | Converts logits to a probability distribution |
| `nn.Dropout(p)` | Regularization (Section 9) |
| `nn.BatchNorm1d/2d` | Normalizes activations (Section 9) |

**Why activations matter:** ReLU is the default hidden-layer choice because it's cheap and largely avoids the vanishing-gradient problem that plagues sigmoid/tanh in deep networks. Softmax is reserved for the very last layer for multi-class probabilities — and note `nn.CrossEntropyLoss` in PyTorch already applies softmax internally, so you should NOT put a softmax before it (Section 4).

In [16]:
# nn.Sequential — quick way to stack layers when you don't need custom forward logic
seq_model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 3),
)
print(seq_model)

# Inspect parameters
for name, param in model.named_parameters():
    print(name, param.shape, "trainable:", param.requires_grad)

Sequential(
  (0): Linear(in_features=10, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=3, bias=True)
)
fc1.weight torch.Size([32, 10]) trainable: True
fc1.bias torch.Size([32]) trainable: True
fc2.weight torch.Size([32, 32]) trainable: True
fc2.bias torch.Size([32]) trainable: True
fc3.weight torch.Size([3, 32]) trainable: True
fc3.bias torch.Size([3]) trainable: True


## 4. Loss Functions & Optimizers

**Concept:** The **loss function** measures how wrong the model's predictions are. The **optimizer** uses the gradients of that loss (from autograd) to update the weights so the loss goes down. Together, "compute loss → backward → optimizer step" is the heartbeat of training.

### Common losses
| Task | Loss | Notes |
|---|---|---|
| Multi-class classification | `nn.CrossEntropyLoss` | Expects raw logits + integer class labels. Internally does log-softmax + NLL. |
| Binary classification | `nn.BCEWithLogitsLoss` | Expects raw logits + 0/1 float labels. More numerically stable than `BCELoss`+`Sigmoid`. |
| Regression | `nn.MSELoss` | Mean squared error |
| Regression (robust) | `nn.L1Loss` / `nn.SmoothL1Loss` | Less sensitive to outliers than MSE |

In [28]:
# Classification example
logits = torch.randn(4, 3)          # 4 samples, 3 classes (raw scores, NOT softmaxed)
labels = torch.tensor([0, 2, 1, 1]) # true class indices

ce_loss_fn = nn.CrossEntropyLoss()
loss = ce_loss_fn(logits, labels)
print("cross-entropy loss:", loss.item())

# Regression example
preds = torch.randn(4, 1)
targets = torch.randn(4, 1)
mse_loss_fn = nn.MSELoss()
print("mse loss:", mse_loss_fn(preds, targets).item())

cross-entropy loss: 1.8573603630065918
mse loss: 1.0976293087005615


### Optimizers

**Concept:** Plain gradient descent updates `w = w - lr * grad`. In practice we use variants that adapt the step size and add momentum for faster, more stable convergence.

| Optimizer | When to use |
|---|---|
| `SGD` (+momentum) | Simple, well-understood, often best generalization with tuning |
| `Adam` | Adaptive learning rate, great default for most problems, fast convergence |
| `AdamW` | Adam with *decoupled* weight decay — preferred over Adam when using weight decay |
| `RMSprop` | Common in older RNN literature |

In [30]:
model = SimpleNN(10, 32, 3)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# alternative: torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
# alternative: torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## 5. The Training Loop

**Concept:** Every PyTorch training loop follows the same 5 steps per batch:

1. **Forward pass** — `outputs = model(inputs)`
2. **Compute loss** — `loss = loss_fn(outputs, targets)`
3. **Zero old gradients** — `optimizer.zero_grad()`
4. **Backward pass** — `loss.backward()` (fills `.grad` on every parameter)
5. **Update weights** — `optimizer.step()`

If you memorize nothing else from PyTorch, memorize this loop.

In [ ]:
# Toy regression problem: learn y = 3x + 2 + noise
torch.manual_seed(0)
X = torch.linspace(-5, 5, 200).unsqueeze(1)
y = 3 * X + 2 + torch.randn_like(X) * 0.5

model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

n_epochs = 200
for epoch in range(n_epochs):
    # 1. forward
    preds = model(X)
    # 2. loss
    loss = loss_fn(preds, y)
    # 3. zero grads
    optimizer.zero_grad()
    # 4. backward
    loss.backward()
    # 5. update
    optimizer.step()

    if (epoch + 1) % 40 == 0:
        print(f"epoch {epoch+1:3d} | loss {loss.item():.4f}")

w, b = model.weight.item(), model.bias.item()
print(f"learned: y = {w:.2f}x + {b:.2f}   (target: y = 3x + 2)")

### `model.train()` vs `model.eval()`
Layers like `Dropout` and `BatchNorm` behave differently during training vs inference. Always call:
- `model.train()` before a training loop
- `model.eval()` (+ `torch.no_grad()`) before validation/inference

Forgetting `.eval()` is one of the most common PyTorch bugs — it silently keeps dropout active and uses batch statistics instead of running statistics for BatchNorm.

## 6. Datasets & DataLoaders

**Concept:** `Dataset` defines *how to get one sample*; `DataLoader` wraps a `Dataset` to handle **batching**, **shuffling**, and **parallel loading**. This separation lets you plug any data source into the same training loop.

Training in **mini-batches** (rather than one sample or the whole dataset at once) balances gradient noise (helps escape sharp minima, adds regularization) against computational efficiency (vectorized GPU ops).

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

X = torch.randn(1000, 10)
y = torch.randint(0, 3, (1000,))

dataset = MyDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)

for batch_idx, (xb, yb) in enumerate(loader):
    print("batch", batch_idx, "x:", xb.shape, "y:", yb.shape)
    if batch_idx == 2:
        break

> **`num_workers`**: number of subprocesses for parallel data loading. Use `0` in notebooks/Windows to avoid multiprocessing issues; use `>0` (e.g. 4) in scripts for real speedups on larger datasets.
>
> **`torchvision.datasets`** provides ready-made datasets (MNIST, CIFAR10, ImageNet, etc.) plus `transforms` for preprocessing — see the next section.

## 7. Full Pipeline: Training a Neural Network on Image Data

Putting it all together on MNIST (handwritten digits). This cell downloads the dataset the first time you run it (needs internet).

In [ ]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),                      # PIL image -> tensor, scales to [0,1]
    transforms.Normalize((0.1307,), (0.3081,))  # standardize using MNIST's mean/std
])

train_data = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data  = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=256, shuffle=False)

print("train size:", len(train_data), "| test size:", len(test_data))
xb, yb = next(iter(train_loader))
print("batch shape:", xb.shape, "labels shape:", yb.shape)  # (64, 1, 28, 28)

In [ ]:
class MNIST_MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()             # (B, 1, 28, 28) -> (B, 784)
        self.net = nn.Sequential(
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),                  # 10 digit classes
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MNIST_MLP().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

n_epochs = 3
for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)

    train_loss = running_loss / len(train_data)
    test_acc = evaluate(model, test_loader)
    print(f"epoch {epoch+1}/{n_epochs} | train_loss {train_loss:.4f} | test_acc {test_acc:.4f}")

## 8. Convolutional Neural Networks (CNNs)

**Why not just use `nn.Linear` everywhere?** A fully-connected layer treats every pixel independently and ignores spatial structure, and it needs a huge number of parameters for images (e.g. a 224×224×3 image flattened is ~150K inputs). CNNs exploit two facts about images:

1. **Local patterns matter** (edges, textures) → use small filters that look at local neighborhoods.
2. **The same pattern can appear anywhere** → reuse (share) the same filter weights across the whole image (**parameter sharing** → far fewer parameters, and **translation invariance**).

### Core building blocks

- **Convolution (`nn.Conv2d`)**: slides a small learnable filter (kernel) over the input, computing a weighted sum at each position → produces a *feature map*. Key params:
  - `kernel_size`: filter size (e.g. 3×3)
  - `stride`: step size the filter moves — larger stride = smaller output, less overlap
  - `padding`: zero-pixels added around the border — `padding=1` with `kernel_size=3` keeps output spatial size equal to input ("same" padding)
  - `out_channels`: number of different filters = number of output feature maps (each filter learns to detect a different pattern)

- **Pooling (`nn.MaxPool2d`)**: downsamples feature maps (e.g. takes the max over each 2×2 block), reducing spatial size and giving a small amount of translation invariance, with no learnable parameters.

- **Output size formula**: `out = floor((in + 2*padding - kernel_size) / stride) + 1`

- **Receptive field**: as you stack conv layers, each output "sees" a progressively larger region of the original image — early layers learn edges/colors, deeper layers learn textures, and the deepest layers learn whole object parts.

In [ ]:
# Visualizing what a single conv layer does to shapes
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)
x = torch.randn(1, 1, 28, 28)   # (batch, channels, height, width)
out = conv(x)
print("input :", x.shape)
print("output:", out.shape)     # channels change (1->8), spatial size unchanged (padding=1)

pool = nn.MaxPool2d(kernel_size=2, stride=2)
pooled = pool(out)
print("after pool:", pooled.shape)  # spatial size halved

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # (B,1,28,28) -> (B,32,28,28)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # -> (B,32,14,14)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # -> (B,64,14,14)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # -> (B,64,7,7)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn = SimpleCNN().to(device)
print(cnn)

# quick sanity check
dummy = torch.randn(4, 1, 28, 28).to(device)
print("output shape:", cnn(dummy).shape)  # (4, 10)

In [ ]:
# Train the CNN on MNIST — same training loop pattern as before
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

n_epochs = 2  # keep short for demo purposes; increase for real training
for epoch in range(n_epochs):
    cnn.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = cnn(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)

    train_loss = running_loss / len(train_data)
    test_acc = evaluate(cnn, test_loader)
    print(f"epoch {epoch+1}/{n_epochs} | train_loss {train_loss:.4f} | test_acc {test_acc:.4f}")

## 9. Regularization: Fighting Overfitting

**Concept:** Overfitting = the model memorizes training data instead of learning generalizable patterns (train loss keeps dropping, val loss stops improving or rises). Regularization techniques trade a little training performance for better generalization.

| Technique | How it works |
|---|---|
| **Dropout** (`nn.Dropout(p)`) | Randomly zeroes out a fraction `p` of activations *during training only* → forces redundant, more robust representations. Disabled automatically in `model.eval()`. |
| **Batch Normalization** (`nn.BatchNorm1d/2d`) | Normalizes layer inputs to zero mean/unit variance per mini-batch, then applies a learnable scale/shift. Stabilizes and speeds up training, and also has a mild regularizing effect. Uses batch statistics in train mode, running averages in eval mode. |
| **Weight decay** (L2 regularization) | Adds a penalty proportional to the squared weight magnitude to the loss, discouraging large weights. Set via `optimizer = AdamW(params, weight_decay=1e-2)`. |
| **Data augmentation** | Randomly transforms training images (flips, crops, rotations, color jitter) so the model sees more variety and can't just memorize exact pixels. |
| **Early stopping** | Stop training when validation loss stops improving, even if training loss keeps dropping. |

In [ ]:
# Data augmentation example with torchvision transforms
augment_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),     # NOTE: don't use horizontal flip for digits like 6/9! shown for illustration
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

# Weight decay example
optimizer = torch.optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=1e-2)

# Dropout + BatchNorm are already in SimpleCNN above (see nn.Dropout(0.3), nn.BatchNorm2d)
print("Dropout is active in train mode:", cnn.training)
cnn.eval()
print("Dropout is disabled in eval mode:", not cnn.training)
cnn.train()  # switch back before continuing training

## 10. Learning Rate Scheduling

**Concept:** A fixed learning rate is rarely optimal for the whole training run — a larger LR helps early on (faster progress), while a smaller LR later helps fine-tune and settle into a good minimum. Schedulers adjust `lr` automatically during training.

| Scheduler | Behavior |
|---|---|
| `StepLR` | Multiply LR by `gamma` every `step_size` epochs |
| `CosineAnnealingLR` | Smoothly decays LR following a cosine curve — very popular default |
| `ReduceLROnPlateau` | Reduce LR when a monitored metric (e.g. val loss) stops improving |
| `OneCycleLR` | Ramp LR up then down within one run — often speeds up convergence significantly |

In [ ]:
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

for epoch in range(5):
    # ... train_one_epoch(...) would go here ...
    scheduler.step()   # call once per epoch (some schedulers step per-batch — check docs)
    print(f"epoch {epoch+1}, lr = {optimizer.param_groups[0]['lr']:.6f}")

## 11. Transfer Learning with Pretrained Models

**Concept:** Training a large CNN from scratch needs huge datasets and compute. Instead, start from a model **pretrained on ImageNet** (which already learned general visual features — edges, textures, shapes) and adapt it to your task. Two common strategies:

1. **Feature extraction**: freeze all pretrained layers, replace and train only the final classifier layer.
2. **Fine-tuning**: unfreeze some/all layers and continue training at a small learning rate, letting the model adapt its features to your specific data.

In [ ]:
import torchvision.models as models

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Strategy 1: freeze everything except the final layer
for param in resnet.parameters():
    param.requires_grad = False

num_classes = 10
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)  # replace classifier head (this new layer is trainable by default)

# Only the new head's params will be updated
optimizer = torch.optim.Adam(resnet.fc.parameters(), lr=1e-3)

# Strategy 2: fine-tune the last block too, with a smaller LR for pretrained params
for param in resnet.layer4.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam([
    {"params": resnet.layer4.parameters(), "lr": 1e-5},   # small LR for pretrained weights
    {"params": resnet.fc.parameters(), "lr": 1e-3},        # larger LR for new head
])

print(resnet.fc)

> Pretrained models expect ImageNet-style preprocessing: `transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])` and (usually) 224×224 RGB input.

## 12. GPU / Device Management

**Concept:** Tensors and models must be on the **same device** to interact. GPUs parallelize the matrix operations that dominate deep learning, giving 10-100x speedups for large models.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = SimpleNN(10, 32, 3).to(device)   # moves all parameters to device
x = torch.randn(4, 10).to(device)        # move input to same device
out = model(x)                           # now both are on the same device, this works

# Common bug: forgetting to move either the model or the data
# RuntimeError: Expected all tensors to be on the same device...

# Move back to CPU (e.g. before converting to NumPy)
out_cpu = out.detach().cpu().numpy()

## 13. Saving & Loading Models

**Concept:** The recommended approach is saving the **`state_dict`** (a dict of parameter tensors), not the whole model object — it's more portable and doesn't break if your code changes slightly.

In [ ]:
# Save
torch.save(model.state_dict(), "model_weights.pth")

# Load — must recreate the same architecture first
loaded_model = SimpleNN(10, 32, 3)
loaded_model.load_state_dict(torch.load("model_weights.pth", map_location=device))
loaded_model.to(device)
loaded_model.eval()  # remember this before inference!

# Saving a full checkpoint (weights + optimizer state + epoch) to resume training later
checkpoint = {
    "epoch": 10,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": 0.123,
}
torch.save(checkpoint, "checkpoint.pth")

ckpt = torch.load("checkpoint.pth", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
optimizer.load_state_dict(ckpt["optimizer_state_dict"])
start_epoch = ckpt["epoch"]
print("resuming from epoch", start_epoch)

## 14. Advanced Training Techniques

### Mixed precision training
**Concept:** Use 16-bit floats for most ops (faster, less memory) while keeping a 32-bit master copy for numerical stability where it matters. Big speedups on modern GPUs with minimal accuracy loss.

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

for xb, yb in train_loader:
    xb, yb = xb.to(device), yb.to(device)
    optimizer.zero_grad()

    with torch.autocast(device_type=device.type, enabled=torch.cuda.is_available()):
        logits = cnn(xb)
        loss = loss_fn(logits, yb)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    break  # just demonstrating one step here

### Gradient clipping
Prevents exploding gradients (common in RNNs, or unstable early training) by capping the gradient norm before the optimizer step.

In [ ]:
loss.backward()
torch.nn.utils.clip_grad_norm_(cnn.parameters(), max_norm=1.0)
optimizer.step()

### Gradient accumulation
Simulates a larger batch size than fits in memory by accumulating gradients over several mini-batches before calling `optimizer.step()`.

In [ ]:
accumulation_steps = 4
optimizer.zero_grad()

for i, (xb, yb) in enumerate(train_loader):
    xb, yb = xb.to(device), yb.to(device)
    logits = cnn(xb)
    loss = loss_fn(logits, yb) / accumulation_steps   # scale down since we're summing
    loss.backward()

    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
    if i == 8:
        break  # demo only

## 15. A Peek Beyond CNNs: RNNs & Transformers

CNNs excel at grid-like data (images). For **sequential** data (text, time series, audio), two families dominate:

### Recurrent Neural Networks (RNNs / LSTM / GRU)
**Concept:** Process a sequence one step at a time, maintaining a **hidden state** that summarizes everything seen so far. Plain RNNs suffer from vanishing gradients over long sequences; **LSTM** and **GRU** add gating mechanisms to control what information is kept or forgotten, mitigating this.

In [ ]:
rnn = nn.LSTM(input_size=10, hidden_size=20, num_layers=2, batch_first=True)
x = torch.randn(4, 15, 10)   # (batch, sequence_length, input_size)
output, (h_n, c_n) = rnn(x)
print("output (all timesteps):", output.shape)  # (4, 15, 20)
print("final hidden state:", h_n.shape)           # (num_layers, batch, hidden_size)

### Transformers & Attention
**Concept:** Instead of processing sequentially, **self-attention** lets every position look at every other position directly and learn how much to "attend" to each — this parallelizes well and captures long-range dependencies better than RNNs. This is the architecture behind BERT, GPT, and virtually all modern large language models.

`nn.TransformerEncoderLayer` implements a full multi-head self-attention + feed-forward block:

In [ ]:
encoder_layer = nn.TransformerEncoderLayer(d_model=32, nhead=4, batch_first=True)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

x = torch.randn(4, 15, 32)   # (batch, sequence_length, embedding_dim)
out = transformer_encoder(x)
print("transformer output:", out.shape)  # (4, 15, 32) -- same shape, contextualized

> Going deeper into RNNs/Transformers (embeddings, positional encoding, multi-head attention math, encoder-decoder architectures) is a natural next notebook once this foundation feels solid.

## 16. Debugging Checklist & Cheat Sheet

### If loss is `NaN`
- Lower the learning rate
- Add gradient clipping
- Check for `log(0)` / division by zero in custom losses
- Check input data for NaNs/Infs

### If the model isn't learning (loss flat)
- Confirm `optimizer.zero_grad()` is called each step
- Confirm `loss.backward()` and `optimizer.step()` are both called
- Check the learning rate isn't too small
- Verify labels/targets are correctly aligned with inputs
- Make sure you didn't accidentally freeze all parameters (`requires_grad=False`)

### If train acc is high but val acc is low (overfitting)
- Add dropout / weight decay / data augmentation
- Reduce model size or add early stopping
- Get more training data if possible

### Shape mismatch errors
- Print `.shape` at every step during debugging
- Remember `nn.CrossEntropyLoss` wants raw logits `(B, C)` + integer labels `(B,)` — not one-hot, not softmaxed

### General workflow cheat sheet
```python
model.train()
for xb, yb in train_loader:
    xb, yb = xb.to(device), yb.to(device)
    optimizer.zero_grad()
    preds = model(xb)
    loss = loss_fn(preds, yb)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    for xb, yb in val_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb)
        # compute metrics
```

### Where to go next
- Object detection / segmentation (Faster R-CNN, U-Net)
- Sequence modeling in depth (attention math, positional encodings, encoder-decoder)
- Generative models (GANs, VAEs, Diffusion models)
- `torch.compile` and production deployment (TorchScript, ONNX, TorchServe)
